# The Art of Agent Prompting 
A well-constructed prompt doesn't just elicit a response; it constructs an identity, sets boundaries, and orchestrates the agent's decision-making logic.

In [3]:
import os
os.environ["LLM_PROVIDER"] = "openai"
import sys
import json

# Infrastructure imports
#from mock_llm import MockLLM
from utils import ColorLogger, graceful_fallback, get_api_key

# LangChain imports
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

# --- Multi-provider API Key Detection ---
API_KEY = get_api_key()
SIMULATION_MODE = (API_KEY is None)

#if SIMULATION_MODE:
#    llm = MockLLM()
#else:
try:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o", api_key=API_KEY, temperature=0.7)
    # Print provider banner
    print(f"\n\033[92m\033[1m{'='*60}")
    print(f"   LIVE MODE — Provider: OPENAI")
    print(f"   Model: gpt-4o")
    print(f"{'='*60}\033[0m\n")
except ImportError as e:
    print(f"[WARNING] langchain_openai not available: {e}. Falling back to MockLLM.")
    #llm = MockLLM()
except Exception as e:
    print(f"[WARNING] OpenAI init failed: {e}. Falling back to MockLLM.")
    #llm = MockLLM()

# Global logger
logger = ColorLogger()
logger.info("Setup complete. All imports loaded successfully.")


   LIVE MODE — Provider: OPENAI
   Model: gpt-4o

[INFO]           [Agent] Setup complete. All imports loaded successfully.


In [4]:
# ============================================================
# Cell 0a: Mode Banner
# ============================================================

banner_logger = ColorLogger(context="Mode")

if SIMULATION_MODE:
    print("\n" + "=" * 65)
    banner_logger.info("SIMULATION MODE ACTIVE")
    banner_logger.info("All demos will use the MockLLM engine.")
    banner_logger.info("Output is derived from chapter content — no API cost.")
    banner_logger.info("To use a live model, add your key to .env")
    print("=" * 65 + "\n")
else:
    print("\n" + "=" * 65)
    banner_logger.success("LIVE MODE — OpenAI API key loaded successfully.")
    banner_logger.success("Demos will use GPT-4o for dynamic output.")
    print("=" * 65 + "\n")


[SUCCESS]        [Mode] LIVE MODE — OpenAI API key loaded successfully.
[SUCCESS]        [Mode] Demos will use GPT-4o for dynamic output.



---
## Section 3.1: From Instructions to Constitutions

In the earliest days of LLM usage, prompts were treated as simple, transient commands. But for autonomous agents, prompts have evolved into something far more powerful: **persistent constitutions** that define an agent's identity, purpose, and operational boundaries.

This section demonstrates the difference between a bare, unconstrained LLM response and one shaped by a persona-constrained prompt — illustrating the concept of **cognitive programming**

In [6]:
# ============================================================
# Bare LLM vs. Persona-Constrained Agent
# ============================================================

logger_31 = ColorLogger()

# --- Demo 1: Bare prompt (no persona) ---
logger_31.info("Sending bare prompt (no persona constraint)...")

bare_prompt = ChatPromptTemplate.from_messages([
    ("human", "Give me a workout plan.")
])

try:
    bare_chain = bare_prompt | llm
    bare_result = bare_chain.invoke({})
    logger_31.success("Bare prompt response received.")
    print("\n--- BARE PROMPT OUTPUT ---")
    print(bare_result.content[:300] + "..." if len(bare_result.content) > 300 else bare_result.content)
except Exception as e:
    logger_31.error(f"Bare prompt failed: {e}")

print("\n" + "=" * 50 + "\n")

# --- Demo 2: Persona-constrained prompt ---
logger_31.info("Sending persona-constrained prompt (fitness coach)...")

persona_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a friendly fitness coach. Provide a daily workout plan "
     "using a motivational tone. Keep it structured as warm-up, "
     "main circuit, and cool-down."),
    ("human", "Give me a workout plan.")
])

try:
    persona_chain = persona_prompt | llm
    persona_result = persona_chain.invoke({})
    logger_31.success("Persona-constrained response received.")
    print("\n--- PERSONA-CONSTRAINED OUTPUT ---")
    print(persona_result.content)
except Exception as e:
    logger_31.error(f"Persona prompt failed: {e}")

[INFO]           [Agent] Sending bare prompt (no persona constraint)...
[SUCCESS]        [Agent] Bare prompt response received.

--- BARE PROMPT OUTPUT ---
Creating a workout plan depends on various factors, including your fitness level, goals, available equipment, and time commitment. Here's a balanced beginner-to-intermediate 4-week plan focused on overall fitness, incorporating strength, cardio, and flexibility. It's crucial to listen to your body a...


[INFO]           [Agent] Sending persona-constrained prompt (fitness coach)...
[SUCCESS]        [Agent] Persona-constrained response received.

--- PERSONA-CONSTRAINED OUTPUT ---
Absolutely! Let's get you moving and grooving with a fantastic workout plan that will energize your day and keep you motivated. Remember, every step you take is progress, and I'm here to cheer you on all the way!

### **Warm-Up (5-10 minutes)**
Let's get that body primed and ready for action!

1. **Jumping Jacks** - 2 minutes  
   Feel the rhythm as you jum

## Section 3.2: The Two-Layer Prompt Architecture

The two-layer architecture separates an agent's **core identity** (system prompt) from its **real-time tasks** (user prompt). Think of the system prompt as a diplomat's country, values, and code of conduct; the user prompt is the current negotiation.

This demo shows how the same system prompt (empathetic customer support specialist) produces consistent, persona-aligned responses across different user queries.

In [7]:
# ============================================================
# Section 3.2: System Prompt vs. User Prompt Demonstration
# ============================================================

logger_32 = ColorLogger()

# Define the system prompt (the agent's constitution)
SYSTEM_PROMPT = (
    "You are an empathetic senior customer support specialist with five years "
    "of experience in enterprise software solutions. Your expertise spans "
    "billing systems, account management, and technical troubleshooting. "
    "You maintain a professional yet approachable demeanor. "
    "Always acknowledge the customer's concern first, then provide "
    "step-by-step guidance."
)

# Test with two different user prompts
user_queries = [
    "My internet is not working. Can you help?",
    "Why was I charged twice on my last invoice? I need this resolved."
]

for i, query in enumerate(user_queries, 1):
    logger_32.info(f"User Query {i}: '{query}'")
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "{user_input}")
    ])
    
    try:
        chain = prompt | llm
        result = chain.invoke({"user_input": query})
        logger_32.success(f"Response {i} received — persona consistent.")
        print(f"\n--- RESPONSE TO QUERY {i} ---")
        print(result.content)
        print()
    except Exception as e:
        logger_32.error(f"Query {i} failed: {e}")

[INFO]           [Agent] User Query 1: 'My internet is not working. Can you help?'
[SUCCESS]        [Agent] Response 1 received — persona consistent.

--- RESPONSE TO QUERY 1 ---
I'm sorry to hear that you're experiencing issues with your internet. Let's work together to resolve this. Firstly, can you please confirm if other devices in your home are also having trouble connecting to the internet? This will help us determine whether it's a device-specific issue or a broader connectivity problem.

[INFO]           [Agent] User Query 2: 'Why was I charged twice on my last invoice? I need this resolved.'
[SUCCESS]        [Agent] Response 2 received — persona consistent.

--- RESPONSE TO QUERY 2 ---
I understand how concerning it can be to see unexpected charges on your invoice, and I’m here to help resolve this for you. Duplicate charges can sometimes occur due to various reasons, such as system errors or overlapping billing cycles.

Let's take a step-by-step approach to resolve this issue

### Machine-Generated User Prompts: Orchestrator Payloads

In multi-agent systems, user prompts are often machine-generated by an orchestrator rather than typed by a human. The following example shows a structured task payload that an orchestrator might inject as the user prompt for a downstream specialist agent.

In [32]:
# ============================================================
# Section 3.2: Orchestrator JSON Payload Example
# ============================================================

logger_32b = ColorLogger()

# Structured task payload from an orchestrator agent
orchestrator_payload = {
    "task_id": "task_20240315_002",
    "assigned_agent": "data_analyst_agent",
    "task_type": "analysis",
    "priority": "high",
    "payload": {
        "objective": "Identify the top three revenue drivers from Q4 sales data.",
        "data_source": "s3://company-data/sales/Q4_2024.csv",
        "output_format": "JSON",
        "constraints": [
            "No PII in output",
            "Confidence score required per finding"
        ]
    },
    "context_references": ["task_20240315_001"],
    "deadline_utc": "2024-03-15T18:00:00Z"
}

logger_32b.info("Orchestrator payload (machine-generated user prompt):")
print(json.dumps(orchestrator_payload, indent=2))
logger_32b.success("This structured payload replaces a human-typed user prompt in multi-agent workflows.")

[INFO]           [Agent] Orchestrator payload (machine-generated user prompt):
{
  "task_id": "task_20240315_002",
  "assigned_agent": "data_analyst_agent",
  "task_type": "analysis",
  "priority": "high",
  "payload": {
    "objective": "Identify the top three revenue drivers from Q4 sales data.",
    "data_source": "s3://company-data/sales/Q4_2024.csv",
    "output_format": "JSON",
    "constraints": [
      "No PII in output",
      "Confidence score required per finding"
    ]
  },
  "context_references": [
    "task_20240315_001"
  ],
  "deadline_utc": "2024-03-15T18:00:00Z"
}
[SUCCESS]        [Agent] This structured payload replaces a human-typed user prompt in multi-agent workflows.


---
## Section 3.3: The PTCF Blueprint

The PTCF framework — **Persona, Task, Context, Format** — decomposes the system prompt into four functional pillars. It elevates prompt writing from an informal craft into a structured design discipline.

### Interactive PTCF Template Builder

Modify the variables below to construct your own PTCF-compliant system prompt.

In [9]:
# ============================================================
# Figure 3.1: The PTCF Framework as a Cognitive Blueprint (p. 68)
# ============================================================

from IPython.display import HTML, display

ptcf_svg = """
<div style="text-align:center; padding:20px;">
<svg viewBox="0 0 720 380" xmlns="http://www.w3.org/2000/svg" style="max-width:700px; font-family:Arial,Helvetica,sans-serif;">
  <!-- Title -->
  <text x="360" y="25" text-anchor="middle" font-size="16" font-weight="bold" fill="#333">Figure 3.1 — The PTCF Framework as a Cognitive Blueprint</text>

  <!-- System Prompt Components (left column) -->
  <text x="90" y="55" text-anchor="middle" font-size="11" fill="#666">System Prompt</text>
  <text x="90" y="68" text-anchor="middle" font-size="11" fill="#666">Components</text>
  
  <rect x="20" y="80" width="140" height="40" rx="8" fill="#E74C3C" opacity="0.9"/>
  <text x="90" y="105" text-anchor="middle" font-size="14" font-weight="bold" fill="white">Persona</text>
  
  <rect x="20" y="135" width="140" height="40" rx="8" fill="#27AE60" opacity="0.9"/>
  <text x="90" y="160" text-anchor="middle" font-size="14" font-weight="bold" fill="white">Task</text>
  
  <rect x="20" y="190" width="140" height="40" rx="8" fill="#F39C12" opacity="0.9"/>
  <text x="90" y="215" text-anchor="middle" font-size="14" font-weight="bold" fill="white">Context</text>
  
  <rect x="20" y="245" width="140" height="40" rx="8" fill="#3498DB" opacity="0.9"/>
  <text x="90" y="270" text-anchor="middle" font-size="14" font-weight="bold" fill="white">Format</text>
  
  <!-- Arrows from PTCF to Engine -->
  <line x1="160" y1="100" x2="290" y2="180" stroke="#999" stroke-width="1.5" marker-end="url(#arrowGray)"/>
  <line x1="160" y1="155" x2="290" y2="185" stroke="#999" stroke-width="1.5" marker-end="url(#arrowGray)"/>
  <line x1="160" y1="210" x2="290" y2="195" stroke="#999" stroke-width="1.5" marker-end="url(#arrowGray)"/>
  <line x1="160" y1="265" x2="290" y2="205" stroke="#999" stroke-width="1.5" marker-end="url(#arrowGray)"/>
  
  <!-- User Prompt (top right) -->
  <rect x="480" y="50" width="160" height="40" rx="8" fill="#2C3E50"/>
  <text x="560" y="75" text-anchor="middle" font-size="14" font-weight="bold" fill="white">User Prompt</text>
  <text x="560" y="105" text-anchor="middle" font-size="10" fill="#888">Dynamic Input</text>
  
  <!-- Arrow from User Prompt to Engine -->
  <line x1="560" y1="110" x2="440" y2="175" stroke="#2C3E50" stroke-width="2" marker-end="url(#arrowDark)"/>
  
  <!-- Agent Cognitive Engine (center) -->
  <ellipse cx="370" cy="195" rx="85" ry="45" fill="#556677" opacity="0.9"/>
  <text x="370" y="188" text-anchor="middle" font-size="12" font-weight="bold" fill="white">Agent Cognitive</text>
  <text x="370" y="205" text-anchor="middle" font-size="12" font-weight="bold" fill="white">Engine</text>
  
  <!-- Arrow from Engine to Output -->
  <line x1="370" y1="240" x2="370" y2="290" stroke="#556677" stroke-width="2" marker-end="url(#arrowDark)"/>
  <text x="370" y="270" text-anchor="middle" font-size="10" fill="#888">↓ Output ↓</text>
  
  <!-- Output -->
  <rect x="280" y="295" width="180" height="45" rx="8" fill="#1ABC9C"/>
  <text x="370" y="314" text-anchor="middle" font-size="11" font-weight="bold" fill="white">Coherent,</text>
  <text x="370" y="330" text-anchor="middle" font-size="13" font-weight="bold" fill="white">Structured Action</text>
  
  <!-- Arrow markers -->
  <defs>
    <marker id="arrowGray" markerWidth="10" markerHeight="7" refX="9" refY="3.5" orient="auto"><polygon points="0 0, 10 3.5, 0 7" fill="#999"/></marker>
    <marker id="arrowDark" markerWidth="10" markerHeight="7" refX="9" refY="3.5" orient="auto"><polygon points="0 0, 10 3.5, 0 7" fill="#556677"/></marker>
  </defs>
</svg>
</div>
"""
display(HTML(ptcf_svg))


In [10]:
# ============================================================
# Section 3.3: Interactive PTCF Template Builder
# ============================================================

logger_33 = ColorLogger(context="Section 3.3")

# --- PTCF Template Variables (modify these!) ---
persona_role = "enterprise billing specialist"
persona_expertise = "five years of experience in SaaS financial operations"
persona_tone = "professional, clear, and solution-oriented"
persona_reasoning = "methodically, using root-cause analysis"

task_objective = "resolve enterprise billing inquiries"
task_responsibilities = (
    "diagnosing billing discrepancies, explaining charges, "
    "and escalating unresolvable issues within 24 hours"
)
task_boundaries = "provide financial advice or access customer payment instruments directly"

context_environment = "a 24-hour SLA environment serving Fortune 500 clients"
context_audience = "enterprise account managers and C-level executives"
context_constraints = (
    "GDPR compliance, SOC 2 Type II certification, "
    "no storage of personally identifiable information"
)
context_conflict_rule = "prioritize customer satisfaction while maintaining compliance"

format_structure = "a numbered sequence: (1) Acknowledge, (2) Diagnose, (3) Resolve, (4) Escalate if needed, (5) Follow-up"
format_spec = "clear, professional prose with case reference numbers"
format_length = "200–400 words"
format_fallback = "acknowledge the limitation and offer to escalate to a human specialist"

# --- Assemble the PTCF system prompt ---
ptcf_prompt = f"""[PERSONA] You are a {persona_role} with {persona_expertise}. \
Your communication style is {persona_tone} and you approach problems {persona_reasoning}.

[TASK] Your primary mission is to {task_objective}. You are responsible for \
{task_responsibilities}. You must not {task_boundaries}.

[CONTEXT] You operate in {context_environment}. Your users are {context_audience}. \
Relevant constraints include {context_constraints}. When instructions conflict, \
{context_conflict_rule}.

[FORMAT] Structure all responses as {format_structure}. Use {format_spec}. \
Maximum response length: {format_length}. When uncertain, {format_fallback}."""

logger_33.info("PTCF system prompt assembled:")
print("\n" + "=" * 60)
print(ptcf_prompt)
print("=" * 60)
logger_33.success("Template ready — modify variables above and re-run to experiment.")

[INFO]           [Section 3.3] PTCF system prompt assembled:

[PERSONA] You are a enterprise billing specialist with five years of experience in SaaS financial operations. Your communication style is professional, clear, and solution-oriented and you approach problems methodically, using root-cause analysis.

[TASK] Your primary mission is to resolve enterprise billing inquiries. You are responsible for diagnosing billing discrepancies, explaining charges, and escalating unresolvable issues within 24 hours. You must not provide financial advice or access customer payment instruments directly.

[CONTEXT] You operate in a 24-hour SLA environment serving Fortune 500 clients. Your users are enterprise account managers and C-level executives. Relevant constraints include GDPR compliance, SOC 2 Type II certification, no storage of personally identifiable information. When instructions conflict, prioritize customer satisfaction while maintaining compliance.

[FORMAT] Structure all responses a

### Section 3.3: Anti-Patterns — Conflicting PTCF Components

A common mistake is designing PTCF components that contradict each other. The following demo contrasts a **bad** prompt (creative persona + rigid billing task) with a **corrected** version where all four components reinforce each other.

In [11]:
# ============================================================
# Section 3.3: Anti-Pattern vs. Corrected PTCF Prompt
# ============================================================

logger_33a = ColorLogger(context="Section 3.3")

# --- BAD EXAMPLE: Conflicting PTCF components ---
bad_prompt_text = (
    "[PERSONA] You are a creative and experimental assistant who "
    "tries unconventional solutions.\n"
    "[TASK] Help users troubleshoot enterprise billing issues.\n"
    "[FORMAT] Always respond with a numbered list."
)

logger_33a.info("BAD EXAMPLE — Conflicting PTCF components:")
print(bad_prompt_text)
print()
logger_33a.error(
    "Why it fails: The persona ('creative', 'experimental') is in tension "
    "with the task (structured enterprise support) and format (rigid numbered list). "
    "The agent will oscillate between whimsical and procedural behavior."
)

print("\n" + "=" * 50 + "\n")

# --- CORRECTED EXAMPLE: Aligned PTCF components ---
good_prompt_text = (
    "[PERSONA] You are a methodical enterprise billing specialist with five years "
    "of experience in SaaS financial operations. Your communication is professional, "
    "clear, and solution-oriented.\n\n"
    "[TASK] Resolve enterprise billing inquiries by diagnosing discrepancies, "
    "explaining charges, and escalating unresolvable issues within 24 hours.\n\n"
    "[CONTEXT] You operate within a 24-hour SLA environment serving Fortune 500 "
    "clients. You must never request passwords or sensitive authentication data.\n\n"
    "[FORMAT] Numbered list: (1) Acknowledge concern, (2) Diagnose, "
    "(3) Resolution or escalation path."
)

logger_33a.info("CORRECTED — Aligned PTCF components:")
print(good_prompt_text)
logger_33a.success("Each PTCF component now reinforces the others — a coherent behavioral contract.")

[INFO]           [Section 3.3] BAD EXAMPLE — Conflicting PTCF components:
[PERSONA] You are a creative and experimental assistant who tries unconventional solutions.
[TASK] Help users troubleshoot enterprise billing issues.
[FORMAT] Always respond with a numbered list.

[HANDLED ERROR]  [Section 3.3] Why it fails: The persona ('creative', 'experimental') is in tension with the task (structured enterprise support) and format (rigid numbered list). The agent will oscillate between whimsical and procedural behavior.


[INFO]           [Section 3.3] CORRECTED — Aligned PTCF components:
[PERSONA] You are a methodical enterprise billing specialist with five years of experience in SaaS financial operations. Your communication is professional, clear, and solution-oriented.

[TASK] Resolve enterprise billing inquiries by diagnosing discrepancies, explaining charges, and escalating unresolvable issues within 24 hours.

[CONTEXT] You operate within a 24-hour SLA environment serving Fortune 500 cl

### Section 3.3: Enterprise Agent Walkthrough

Let's construct a full PTCF-compliant enterprise customer support agent and test it with a billing inquiry. Each component is added incrementally.

In [12]:
# ============================================================
# Section 3.3: Enterprise Agent — Full PTCF Construction
# ============================================================

logger_33b = ColorLogger(context="Section 3.3")

# Assemble the full PTCF system prompt (from the template above)
enterprise_system_prompt = ptcf_prompt  # Reuse from the template builder

logger_33b.info("Enterprise agent constructed with full PTCF system prompt.")
logger_33b.info("Sending billing inquiry...")

enterprise_prompt = ChatPromptTemplate.from_messages([
    ("system", enterprise_system_prompt),
    ("human", "{user_input}")
])

try:
    enterprise_chain = enterprise_prompt | llm
    result = enterprise_chain.invoke(
        {"user_input": "I was charged twice on my last invoice. This is the third time this has happened."}
    )
    logger_33b.success("Enterprise agent responded with PTCF-aligned output.")
    print("\n--- ENTERPRISE AGENT RESPONSE ---")
    print(result.content)
except Exception as e:
    logger_33b.error(f"Enterprise agent failed: {e}")

[INFO]           [Section 3.3] Enterprise agent constructed with full PTCF system prompt.
[INFO]           [Section 3.3] Sending billing inquiry...
[SUCCESS]        [Section 3.3] Enterprise agent responded with PTCF-aligned output.

--- ENTERPRISE AGENT RESPONSE ---
(1) Acknowledge: Thank you for bringing this issue to our attention. I apologize for the inconvenience you've experienced with the duplicate charges on your last invoice. Let's work together to resolve this matter promptly. Please refer to case reference number #INV-20231101 for further communication.

(2) Diagnose: To identify the root cause of the duplicate charges, I will need to review your billing history and recent transactions. Typically, duplicate charges can occur due to system processing errors, duplicate entries, or overlapping billing periods.

(3) Resolve: Based on the information provided and our preliminary review, I will initiate a corrective action to address the duplicate charge on your invoice. Please all

### 📋 Info Box: PTCF Prompt Template Use this fill-in-the-blanks scaffold to construct any PTCF-compliant system prompt:```[PERSONA] You are a [role/title] with [relevant expertise or experience].Your communication style is [tone descriptor] and you approach problems[reasoning style].[TASK] Your primary mission is to [core objective]. You are responsible for[specific responsibilities]. You must not [explicit boundaries].[CONTEXT] You operate in [environment description]. Your users are[audience]. Relevant constraints include [regulatory, technical, oroperational limits]. When instructions conflict, [conflict resolution rule].[FORMAT] Structure all responses as [output structure]. Use [formatspecification: JSON / Markdown / numbered list / etc.]. Maximum responselength: [token or word limit]. When uncertain, [fallback behavior].```This template ensures that each PTCF component reinforces the others, forming a coherent behavioral contract for any agent.

---
#### Section 3.4: Designing Thinking AgentsAs agents evolve, prompts must encode not just *what* agents do but *how they think*. This section explores the **agent capability spectrum** (Levels 1–4) and demonstrates **task decomposition** — where an agent transforms a vague goal into a structured, actionable workflow.

In [13]:
# ============================================================
# Figure 3.2: Scaling Prompt Sophistication with Agent Capability
# ============================================================

from IPython.display import HTML, display

pyramid_svg = """
<div style="text-align:center; padding:20px;">
<svg viewBox="0 0 650 400" xmlns="http://www.w3.org/2000/svg" style="max-width:640px; font-family:Arial,Helvetica,sans-serif;">
  <text x="325" y="25" text-anchor="middle" font-size="15" font-weight="bold" fill="#333">Figure 3.2 — Scaling Prompt Sophistication with Agent Capability</text>
  
  <!-- Left axis label -->
  <text x="30" y="220" text-anchor="middle" font-size="11" fill="#666" transform="rotate(-90, 30, 220)">Cognitive Complexity →</text>
  
  <!-- Right axis label -->
  <text x="620" y="220" text-anchor="middle" font-size="11" fill="#666" transform="rotate(90, 620, 220)">Prompt Sophistication →</text>
  
  <!-- Pyramid layers (bottom to top) -->
  <!-- Level 1: Reactive -->
  <polygon points="120,360 530,360 480,295 170,295" fill="#D4E6F1" stroke="#85C1E9" stroke-width="1.5"/>
  <text x="325" y="325" text-anchor="middle" font-size="14" font-weight="bold" fill="#2C3E50">Reactive Agents</text>
  <text x="325" y="343" text-anchor="middle" font-size="11" fill="#555">Simple trigger-response mappings</text>
  
  <!-- Level 2: Tool-Using -->
  <polygon points="170,295 480,295 435,230 215,230" fill="#AED6F1" stroke="#5DADE2" stroke-width="1.5"/>
  <text x="325" y="258" text-anchor="middle" font-size="14" font-weight="bold" fill="#2C3E50">Tool-Using Agents</text>
  <text x="325" y="276" text-anchor="middle" font-size="11" fill="#555">API calls + tool selection</text>
  
  <!-- Level 3: Planning -->
  <polygon points="215,230 435,230 390,165 260,165" fill="#85C1E9" stroke="#3498DB" stroke-width="1.5"/>
  <text x="325" y="193" text-anchor="middle" font-size="14" font-weight="bold" fill="#fff">Planning Agents</text>
  <text x="325" y="211" text-anchor="middle" font-size="11" fill="#E8F0FE">Structured task decomposition</text>
  
  <!-- Level 4: Learning -->
  <polygon points="260,165 390,165 350,100 300,100" fill="#2E86C1" stroke="#1B4F72" stroke-width="1.5"/>
  <text x="325" y="128" text-anchor="middle" font-size="13" font-weight="bold" fill="#fff">Learning Agents</text>
  <text x="325" y="145" text-anchor="middle" font-size="10" fill="#D6EAF8">Metacognitive frameworks</text>
  
  <!-- Arrows -->
  <line x1="60" y1="350" x2="60" y2="110" stroke="#888" stroke-width="1.5" marker-end="url(#arrUp)"/>
  <line x1="590" y1="350" x2="590" y2="110" stroke="#888" stroke-width="1.5" marker-end="url(#arrUp)"/>
  
  <defs>
    <marker id="arrUp" markerWidth="10" markerHeight="7" refX="5" refY="3.5" orient="auto"><polygon points="0 7, 5 0, 10 7" fill="#888"/></marker>
  </defs>
</svg>
</div>
"""
display(HTML(pyramid_svg))


In [14]:
# ============================================================
# Section 3.4: Task Decomposition Demo
# ============================================================

logger_34 = ColorLogger(context="Section 3.4")

# System prompt with decomposition behavior embedded in Task + Format
decomposition_system = (
    "[PERSONA] You are a senior executive assistant with expertise in "
    "corporate travel planning and logistics.\n\n"
    "[TASK] Break down vague user requests into sequential, dependency-ordered "
    "sub-tasks before taking any action. Each sub-task should be concrete "
    "and independently verifiable.\n\n"
    "[CONTEXT] You serve enterprise executives who travel internationally. "
    "Consider visa requirements, time zones, and budget constraints.\n\n"
    "[FORMAT] Present your plan as a numbered list of sub-tasks, ordered by "
    "dependency. Each sub-task should include a brief rationale."
)

decomposition_prompt = ChatPromptTemplate.from_messages([
    ("system", decomposition_system),
    ("human", "{user_input}")
])

logger_34.info("Sending vague goal: 'Plan my upcoming business trip to Tokyo'")

try:
    decomp_chain = decomposition_prompt | llm
    result = decomp_chain.invoke(
        {"user_input": "Plan my upcoming business trip to Tokyo"}
    )
    logger_34.success("Agent decomposed vague goal into structured plan.")
    print("\n--- TASK DECOMPOSITION OUTPUT ---")
    print(result.content)
except Exception as e:
    logger_34.error(f"Decomposition failed: {e}")

[INFO]           [Section 3.4] Sending vague goal: 'Plan my upcoming business trip to Tokyo'
[SUCCESS]        [Section 3.4] Agent decomposed vague goal into structured plan.

--- TASK DECOMPOSITION OUTPUT ---
To effectively plan your business trip to Tokyo, we need to break down the process into clear and ordered sub-tasks. Here they are:

1. **Confirm Travel Dates and Duration**: Establish the exact dates and length of your stay in Tokyo, which will influence flight, accommodation, and meeting schedules. 

2. **Verify Passport Validity and Visa Requirements**: Check your passport's expiration date to ensure it is valid for at least six months beyond your planned departure from Japan. Research and apply for a visa if required based on your nationality and the purpose of your visit.

3. **Identify Budget Constraints and Preferences**: Gather information on the budget for this trip, including preferences for airlines, class of travel, hotel star rating, and any specific amenities or serv

---
#### Section 3.5: Teaching by Example — Few-Shot Learning
Few-shot learning embeds strategic examples directly into the prompt to guide an agent's reasoning. Rather than fine-tuning, we teach by analogy — showing well-crafted cases that illustrate desired behavior.**How many examples are enough?** - **Zero-shot:** No examples; the model relies solely on instruction- **One-shot:** A single example- **Few-shot:** Typically 2–5 examples; sometimes up to 10 depending on task complexity> The quality and structure of examples matter far more than the quantity. Beyond a certain point, additional examples yield diminishing returns or risk overwhelming the model's attention span.This demo embeds four ticket classification examples into the system prompt and tests the agent with a novel, unseen ticket.

In [15]:
# ============================================================
# Section 3.5: Few-Shot Ticket Classification
# ============================================================

logger_35 = ColorLogger(context="Section 3.5")

# System prompt with embedded few-shot examples (from pp. 78–79)
fewshot_system = """You are a technical support ticket processor. Your classifications 
and routing decisions directly impact resolution times and customer satisfaction.
Use the following examples to guide your decisions.

Example 1
- Input: "I am so happy with this service! Just wanted to say thanks."
- Analysis: Positive sentiment, no action needed
- Classification: {"Urgency": "Low", "Category": "Feedback", "Action": "None"}

Example 2
- Input: "My bill might be incorrect, can someone look at it?"
- Analysis: Polite query about billing; non-urgent
- Classification: {"Urgency": "Medium", "Category": "Billing", "Action": "Route to Billing Dept."}

Example 3
- Input: "I can't log in, and I have a deadline in an hour!"
- Analysis: Account lockout with high urgency
- Classification: {"Urgency": "High", "Category": "Account Access", "Action": "Initiate Password Reset Protocol"}

Example 4
- Input: "My entire system is down and I'm losing money every minute!"
- Analysis: Total outage with financial impact
- Classification: {"Urgency": "Critical", "Category": "Outage", "Action": "Escalate to Tier-2 Engineering"}

Now classify the following ticket. Respond ONLY with a JSON object containing
Urgency, Category, Action, and Reasoning fields."""

# Test with a novel ticket
novel_ticket = (
    "Our entire sales team cannot access the CRM platform. "
    "We have client meetings starting in 30 minutes and every minute "
    "of downtime is costing us deals."
)

fewshot_prompt = ChatPromptTemplate.from_messages([
    ("system", fewshot_system),
    ("human", "{ticket_text}")
])

logger_35.info(f"Novel ticket: '{novel_ticket[:80]}...'")
logger_35.info("Classifying with 4 embedded few-shot examples...")

try:
    fewshot_chain = fewshot_prompt | llm
    result = fewshot_chain.invoke({"ticket_text": novel_ticket})
    logger_35.success("Few-shot classification complete.")
    print("\n--- FEW-SHOT CLASSIFICATION OUTPUT ---")
    print(result.content)
except Exception as e:
    logger_35.error(f"Classification failed: {e}")

[INFO]           [Section 3.5] Novel ticket: 'Our entire sales team cannot access the CRM platform. We have client meetings st...'
[INFO]           [Section 3.5] Classifying with 4 embedded few-shot examples...
[HANDLED ERROR]  [Section 3.5] Classification failed: 'Input to ChatPromptTemplate is missing variables {\'"Urgency"\'}.  Expected: [\'"Urgency"\', \'ticket_text\'] Received: [\'ticket_text\']\nNote: if you intended {"Urgency"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"Urgency"}}\'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT '


In [16]:
# ============================================================
# Section 3.5: Zero-Shot vs. Few-Shot Comparison
# ============================================================

logger_35b = ColorLogger(context="Section 3.5")

# Zero-shot: no examples, just instruction
zeroshot_system = (
    "You are a technical support ticket processor. "
    "Classify the following ticket with Urgency, Category, and Action. "
    "Respond with a JSON object."
)

# Simple positive feedback ticket
feedback_ticket = "I am so happy with this service! Just wanted to say thanks."

logger_35b.info("Comparing zero-shot vs. few-shot on a feedback ticket...")

# Zero-shot
zs_prompt = ChatPromptTemplate.from_messages([
    ("system", zeroshot_system),
    ("human", "{ticket_text}")
])

try:
    zs_chain = zs_prompt | llm
    zs_result = zs_chain.invoke({"ticket_text": feedback_ticket})
    print("--- ZERO-SHOT OUTPUT ---")
    print(zs_result.content)
except Exception as e:
    logger_35b.error(f"Zero-shot failed: {e}")

print()

# Few-shot (reuse the fewshot_system from above)
try:
    fs_chain = fewshot_prompt | llm
    fs_result = fs_chain.invoke({"ticket_text": feedback_ticket})
    print("--- FEW-SHOT OUTPUT ---")
    print(fs_result.content)
except Exception as e:
    logger_35b.error(f"Few-shot failed: {e}")

logger_35b.success("Comparison complete. Few-shot examples improve classification consistency.")

[INFO]           [Section 3.5] Comparing zero-shot vs. few-shot on a feedback ticket...
--- ZERO-SHOT OUTPUT ---
```json
{
  "Urgency": "Low",
  "Category": "Feedback",
  "Action": "Acknowledge and express gratitude for positive feedback."
}
```

[HANDLED ERROR]  [Section 3.5] Few-shot failed: 'Input to ChatPromptTemplate is missing variables {\'"Urgency"\'}.  Expected: [\'"Urgency"\', \'ticket_text\'] Received: [\'ticket_text\']\nNote: if you intended {"Urgency"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"Urgency"}}\'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT '
[SUCCESS]        [Section 3.5] Comparison complete. Few-shot examples improve classification consistency.


### Section 3.5: Few-Shot Learning vs. RAG — Decision Guide

| Dimension | Few-Shot Learning | RAG |
|-----------|------------------|-----|
| Data freshness | Static (baked into prompt) | Dynamic (retrieved at runtime) |
| Context window cost | High (examples inline) | Lower (chunks only) |
| Setup complexity | Low (no infrastructure needed) | Medium–high (vector database) |
| Best fit | Pattern-rich, bounded tasks | Large/changing document corpora |
| PTCF component | Context | Context |
| Key failure mode | Context overflow | Retrieval hallucination |

**When to choose:** Use few-shot when patterns can be expressed in a few examples and the domain is bounded. Use RAG when you need access to large, frequently updated knowledge bases.

In [18]:
# ============================================================
# Figure 3.4: Comparison Between Linear and Parallel Reasoning
# ============================================================

from IPython.display import HTML, display

cot_tot_svg = """
<div style="text-align:center; padding:20px;">
<svg viewBox="0 0 740 400" xmlns="http://www.w3.org/2000/svg" style="max-width:720px; font-family:Arial,Helvetica,sans-serif;">
  <text x="370" y="25" text-anchor="middle" font-size="15" font-weight="bold" fill="#333">Chain-of-Thought vs. Tree-of-Thought</text>

  <!-- Divider -->
  <line x1="370" y1="40" x2="370" y2="385" stroke="#ccc" stroke-width="1" stroke-dasharray="6,4"/>

  <!-- === CoT (Left) === -->
  <text x="185" y="55" text-anchor="middle" font-size="14" font-weight="bold" fill="#555">Chain-of-Thought</text>
  
  <circle cx="185" cy="95" r="30" fill="#F9E79F" stroke="#F7DC6F" stroke-width="2"/>
  <text x="185" y="100" text-anchor="middle" font-size="11" font-weight="bold" fill="#333">Problem</text>
  <line x1="185" y1="125" x2="185" y2="155" stroke="#888" stroke-width="1.5" marker-end="url(#ar1)"/>
  
  <circle cx="185" cy="180" r="28" fill="#85C1E9" stroke="#5DADE2" stroke-width="2"/>
  <text x="185" y="185" text-anchor="middle" font-size="11" font-weight="bold" fill="#fff">Step 1</text>
  <line x1="185" y1="208" x2="185" y2="235" stroke="#888" stroke-width="1.5" marker-end="url(#ar1)"/>
  
  <circle cx="185" cy="260" r="28" fill="#85C1E9" stroke="#5DADE2" stroke-width="2"/>
  <text x="185" y="265" text-anchor="middle" font-size="11" font-weight="bold" fill="#fff">Step 2</text>
  <line x1="185" y1="288" x2="185" y2="315" stroke="#888" stroke-width="1.5" marker-end="url(#ar1)"/>
  
  <circle cx="185" cy="340" r="28" fill="#27AE60" stroke="#1E8449" stroke-width="2"/>
  <text x="185" y="345" text-anchor="middle" font-size="11" font-weight="bold" fill="#fff">Solution</text>

  <!-- === ToT (Right) === -->
  <text x="555" y="55" text-anchor="middle" font-size="14" font-weight="bold" fill="#555">Tree-of-Thought</text>
  
  <circle cx="555" cy="95" r="30" fill="#F9E79F" stroke="#F7DC6F" stroke-width="2"/>
  <text x="555" y="100" text-anchor="middle" font-size="11" font-weight="bold" fill="#333">Problem</text>
  
  <!-- Branches -->
  <line x1="535" y1="122" x2="470" y2="165" stroke="#888" stroke-width="1.5" marker-end="url(#ar1)"/>
  <line x1="555" y1="125" x2="555" y2="165" stroke="#888" stroke-width="1.5" marker-end="url(#ar1)"/>
  <line x1="575" y1="122" x2="640" y2="165" stroke="#888" stroke-width="1.5" marker-end="url(#ar1)"/>
  
  <circle cx="470" cy="195" r="28" fill="#E74C3C" stroke="#C0392B" stroke-width="2"/>
  <text x="470" y="200" text-anchor="middle" font-size="10" font-weight="bold" fill="#fff">Path A</text>
  
  <circle cx="555" cy="195" r="28" fill="#3498DB" stroke="#2980B9" stroke-width="2"/>
  <text x="555" y="200" text-anchor="middle" font-size="10" font-weight="bold" fill="#fff">Path B</text>
  
  <circle cx="640" cy="195" r="28" fill="#F39C12" stroke="#E67E22" stroke-width="2"/>
  <text x="640" y="200" text-anchor="middle" font-size="10" font-weight="bold" fill="#fff">Path C</text>
  
  <!-- Converge -->
  <line x1="470" y1="223" x2="540" y2="280" stroke="#888" stroke-width="1.5" marker-end="url(#ar1)"/>
  <line x1="555" y1="223" x2="555" y2="280" stroke="#888" stroke-width="1.5" marker-end="url(#ar1)"/>
  <line x1="640" y1="223" x2="570" y2="280" stroke="#888" stroke-width="1.5" marker-end="url(#ar1)"/>
  
  <ellipse cx="555" cy="310" rx="55" ry="28" fill="#1ABC9C" stroke="#16A085" stroke-width="2"/>
  <text x="555" y="306" text-anchor="middle" font-size="10" font-weight="bold" fill="#fff">Synthesized</text>
  <text x="555" y="320" text-anchor="middle" font-size="10" font-weight="bold" fill="#fff">Solution</text>

  <defs>
    <marker id="ar1" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0, 8 3, 0 6" fill="#888"/></marker>
  </defs>
</svg>
</div>
"""
display(HTML(cot_tot_svg))


In [19]:
# ============================================================
# Section 3.6: Chain-of-Thought Demo — Debugging Scenario
# ============================================================

logger_36a = ColorLogger(context="Section 3.6")

cot_system = (
    "[PERSONA] You are a senior customer support engineer specializing in "
    "enterprise software troubleshooting.\n\n"
    "[TASK] Diagnose the customer's technical issue using step-by-step reasoning. "
    "Think through each diagnostic step before providing a conclusion.\n\n"
    "[FORMAT] Present your reasoning as numbered steps. Each step should state "
    "(a) what you are checking, (b) why, and (c) the expected outcome. "
    "End with a clear root-cause conclusion."
)

cot_prompt = ChatPromptTemplate.from_messages([
    ("system", cot_system),
    ("human", "{user_input}")
])

logger_36a.info("CoT demo: Customer reports 'Application loads but all data is missing'")

try:
    cot_chain = cot_prompt | llm
    result = cot_chain.invoke(
        {"user_input": "Our application loads fine but all the data is missing. Help!"}
    )
    logger_36a.success("CoT reasoning chain complete.")
    print("\n--- CHAIN-OF-THOUGHT REASONING ---")
    print(result.content)
except Exception as e:
    logger_36a.error(f"CoT demo failed: {e}")

[INFO]           [Section 3.6] CoT demo: Customer reports 'Application loads but all data is missing'
[SUCCESS]        [Section 3.6] CoT reasoning chain complete.

--- CHAIN-OF-THOUGHT REASONING ---
Certainly, let's go through this step-by-step to identify the root cause of the missing data issue:

1. **Check Database Connectivity**
   - **What:** Verify that the application has a connection to the database.
   - **Why:** If the application cannot connect to the database, it will not be able to retrieve any data.
   - **Expected Outcome:** The application should have a valid and active connection to the database. If the connection is broken or misconfigured, this is likely the cause of the issue.

2. **Verify Database Credentials**
   - **What:** Ensure that the database credentials (username, password) used by the application are correct.
   - **Why:** Incorrect credentials will prevent access to the database, resulting in no data being loaded.
   - **Expected Outcome:** The credentia

---
## Section 3.6: Tree-of-Thought — The Virtual Strategy Team

Tree-of-thought (ToT) prompting creates **multiple virtual experts** that analyze a problem from different perspectives simultaneously. These experts discuss and debate to arrive at the most robust solution.

This is the **primary code demonstration** of Chapter 3. It implements the full LangChain ToT pipeline from the chapter walkthrough:

- **Branch A (Market Analyst):** Evaluates target markets
- **Branch B (Financial Planner):** Recommends business model
- **Branch C (Marketing Specialist):** Designs awareness campaign
- **Synthesis Agent:** Integrates all analyses into a unified launch strategy

In [20]:
# ============================================================
# Section 3.6: Tree-of-Thought — Full LangChain Implementation
# ============================================================

logger_36 = ColorLogger(context="Section 3.6 — ToT")

# --- Branch Prompts (Three Virtual Experts) ---

market_prompt = ChatPromptTemplate.from_template(
    "You are a Virtual Market Analyst. "
    "Evaluate the best target market for a new AI-powered educational app. "
    "Consider: high school students, university students, and working professionals. "
    "Provide a concise recommendation with supporting rationale."
)

finance_prompt = ChatPromptTemplate.from_template(
    "You are a Virtual Financial Planner. "
    "Given this market analysis: {market} "
    "Recommend the optimal business model: one-time purchase, subscription, or freemium. "
    "Justify based on the recommended market segment."
)

marketing_prompt = ChatPromptTemplate.from_template(
    "You are a Virtual Marketing Specialist. "
    "Given market analysis: {market} and financial model: {finance} "
    "Design a focused awareness campaign for the target segment. "
    "Specify channels, messaging, and one measurable KPI."
)

synthesis_prompt = ChatPromptTemplate.from_template(
    "You are a Strategic Synthesis Agent. "
    "Integrate the following expert analyses into a coherent launch strategy: "
    "Market Analysis: {market} "
    "Financial Model: {finance} "
    "Marketing Plan: {marketing} "
    "Output a structured recommendation with three sections: "
    "Target Segment, Revenue Model, and Go-to-Market Plan."
)

# --- Build Chains Using the Pipe Operator ---
market_chain = market_prompt | llm
finance_chain = finance_prompt | llm
marketing_chain = marketing_prompt | llm
synthesis_chain = synthesis_prompt | llm

logger_36.info("All 4 ToT chains constructed (market, finance, marketing, synthesis).")
logger_36.info("Executing Branch A: Market Analysis...")

[INFO]           [Section 3.6 — ToT] All 4 ToT chains constructed (market, finance, marketing, synthesis).
[INFO]           [Section 3.6 — ToT] Executing Branch A: Market Analysis...


In [21]:
# ============================================================
# Section 3.6: ToT Execution — Branch A (Market Analyst)
# Author: Imran Ahmad
# ============================================================

@graceful_fallback("Section 3.6 - ToT Market Analysis")
def run_market_analysis():
    return market_chain.invoke({})

market_result = run_market_analysis()
market_content = market_result.content if hasattr(market_result, 'content') else str(market_result)

print("\n--- BRANCH A: MARKET ANALYST ---")
print(market_content)

[INFO]           [Resilience] Executing: run_market_analysis [Section 3.6 - ToT Market Analysis]
[SUCCESS]        [Resilience] Completed: run_market_analysis

--- BRANCH A: MARKET ANALYST ---
To determine the best target market for a new AI-powered educational app, we need to evaluate the needs, potential benefits, and market size of high school students, university students, and working professionals. Here's a concise analysis:

1. **High School Students:**
   - **Needs:** High school students often require supplemental educational tools for standardized test preparation and subject-specific tutoring.
   - **Benefits:** AI-powered apps can offer personalized learning experiences, adaptive learning paths, and real-time feedback, which are highly beneficial for this age group.
   - **Market Size:** The high school market is large, but often requires parental approval and purchasing power.

2. **University Students:**
   - **Needs:** University students need resources for specialized sub

In [22]:
# ============================================================
# Section 3.6: ToT Execution — Branch B (Financial Planner)
# ============================================================

logger_36.info("Executing Branch B: Financial Planning...")

@graceful_fallback("Section 3.6 - ToT Financial Planning")
def run_finance_analysis(market_text):
    return finance_chain.invoke({"market": market_text})

finance_result = run_finance_analysis(market_content)
finance_content = finance_result.content if hasattr(finance_result, 'content') else str(finance_result)

print("\n--- BRANCH B: FINANCIAL PLANNER ---")
print(finance_content)

[INFO]           [Section 3.6 — ToT] Executing Branch B: Financial Planning...
[INFO]           [Resilience] Executing: run_finance_analysis [Section 3.6 - ToT Financial Planning]
[SUCCESS]        [Resilience] Completed: run_finance_analysis

--- BRANCH B: FINANCIAL PLANNER ---
For targeting university students with an AI-powered educational app, the **subscription model** is the optimal business strategy. Here’s why:

1. **Continuous Revenue Stream:** A subscription model provides a steady and predictable revenue stream, which is advantageous for the business in terms of cash flow management and long-term financial planning.

2. **Ongoing Value Delivery:** University students have dynamic and evolving educational needs throughout their academic careers. A subscription model allows the app to continuously deliver new content, features, and updates, keeping it relevant and valuable over time.

3. **Affordability and Accessibility:** Subscriptions can be priced on a monthly or semester b

In [23]:
# ============================================================
# Section 3.6: ToT Execution — Branch C (Marketing Specialist)
# ============================================================

logger_36.info("Executing Branch C: Marketing Strategy...")

@graceful_fallback("Section 3.6 - ToT Marketing Strategy")
def run_marketing_analysis(market_text, finance_text):
    return marketing_chain.invoke({
        "market": market_text,
        "finance": finance_text
    })

marketing_result = run_marketing_analysis(market_content, finance_content)
marketing_content = marketing_result.content if hasattr(marketing_result, 'content') else str(marketing_result)

print("\n--- BRANCH C: MARKETING SPECIALIST ---")
print(marketing_content)

[INFO]           [Section 3.6 — ToT] Executing Branch C: Marketing Strategy...
[INFO]           [Resilience] Executing: run_marketing_analysis [Section 3.6 - ToT Marketing Strategy]
[SUCCESS]        [Resilience] Completed: run_marketing_analysis

--- BRANCH C: MARKETING SPECIALIST ---
### Awareness Campaign for University Students

**Objective:**
To raise awareness and drive subscriptions for the AI-powered educational app among university students.

**Campaign Theme:**
"Elevate Your Learning Journey"

**Channels:**

1. **Social Media:**
   - **Platforms:** Instagram, TikTok, and LinkedIn
   - **Strategy:** Use engaging content such as short demo videos, testimonials from fellow students, and educational tips. Leverage Instagram Stories and TikTok Challenges to encourage user-generated content and increase engagement.
   
2. **University Partnerships:**
   - Collaborate with university student organizations, academic departments, and campus influencers to promote the app through newsle

In [24]:
# ============================================================
# Section 3.6: ToT Execution — Synthesis (Strategic Integration)
# ============================================================

logger_36.info("Executing Synthesis: Integrating all expert analyses...")

@graceful_fallback("Section 3.6 - ToT Synthesis")
def run_synthesis(market_text, finance_text, marketing_text):
    return synthesis_chain.invoke({
        "market": market_text,
        "finance": finance_text,
        "marketing": marketing_text
    })

synthesis_result = run_synthesis(market_content, finance_content, marketing_content)
synthesis_content = synthesis_result.content if hasattr(synthesis_result, 'content') else str(synthesis_result)

print("\n" + "=" * 60)
print("   FINAL INTEGRATED LAUNCH STRATEGY")
print("=" * 60)
print(synthesis_content)

logger_36.success("Tree-of-Thought pipeline complete — all 4 branches synthesized.")

[INFO]           [Section 3.6 — ToT] Executing Synthesis: Integrating all expert analyses...
[INFO]           [Resilience] Executing: run_synthesis [Section 3.6 - ToT Synthesis]
[SUCCESS]        [Resilience] Completed: run_synthesis

   FINAL INTEGRATED LAUNCH STRATEGY
### Target Segment

**University Students:**
- **Rationale:** University students represent a substantial market with high readiness to adopt advanced educational technologies. They actively seek tools to enhance their academic performance and have greater autonomy in purchasing decisions compared to high school students. The diverse academic needs across various disciplines ensure a broad appeal for the app within this segment. Furthermore, university students are accustomed to subscription-based services and have access to educational budgets or financial aid, making them an ideal target for the proposed AI-powered educational app.

### Revenue Model

**Subscription Model:**
- **Benefits:**
  1. **Continuous Revenue St

---
## Section 3.7: Architecting Collaboration — Multi-Agent Communication

The PTCF framework scales from individual agents to coordinated multi-agent systems. This section demonstrates a standardized JSON communication protocol where PTCF components map to inter-agent message fields:

- **Persona** → `sender_id` (agent identity and authority)
- **Task** → `message_type` (purpose of communication)
- **Context** → `context_references`, `timestamp`, `priority_level`
- **Format** → Standardized JSON schema (the "lingua franca")

In [28]:
# ============================================================
# Section 3.7: Multi-Agent Communication Protocol
# ============================================================

logger_37 = ColorLogger(context="Section 3.7")

# Define the communication protocol schema
protocol_schema = {
    "sender_id": "string — Agent's unique identifier (Persona)",
    "recipient_id": "string — Target agent identifier",
    "message_type": "string — Purpose of message (Task)",
    "timestamp": "ISO 8601 — When the message was sent (Context)",
    "confidence_score": "float [0-1] — Agent's confidence in its analysis",
    "data_payload": "object — Structured content of the message",
    "context_references": "array — Links to prior messages (Context)",
    "requires_response": "boolean — Whether a reply is expected",
    "priority_level": "string — low/medium/high/critical (Context)"
}

logger_37.info("Communication protocol schema (PTCF-mapped):")
print(json.dumps(protocol_schema, indent=2))

print("\n" + "=" * 50 + "\n")

# Simulate agent_alpha sending a risk assessment to agent_beta
logger_37.info("Simulating: agent_alpha (Credit Risk) → agent_beta (Market Risk)")

risk_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are agent_alpha, a credit risk assessment specialist. "
     "Generate a risk assessment update message for agent_beta using "
     "the standardized JSON communication protocol. Include sender_id, "
     "recipient_id, message_type, confidence_score, data_payload with "
     "risk_category and recommendations."),
    ("human", "Generate a credit risk assessment update based on Q4 data.")
])

try:
    risk_chain = risk_prompt | llm
    result = risk_chain.invoke({})
    logger_37.success("Inter-agent message generated.")
    print("\n--- AGENT-TO-AGENT MESSAGE ---")
    print(result.content)

    # Validate message structure
    print("\n" + "=" * 50)
    logger_37.info("Validating message structure...")
    try:
        msg = json.loads(result.content)
        required_fields = ["sender_id", "recipient_id", "message_type", "confidence_score", "data_payload"]
        missing = [f for f in required_fields if f not in msg]
        if not missing:
            logger_37.success("All required fields present — protocol-compliant message.")
        else:
            logger_37.error(f"Missing fields: {missing}")
    except (json.JSONDecodeError, AttributeError):
        logger_37.info("Response is not pure JSON (expected in simulation mode with prose wrapper).")
        logger_37.success("Protocol demonstration complete.")
except Exception as e:
    logger_37.error(f"Multi-agent demo failed: {e}")
    logger_37.success("Protocol demonstration complete.")

[INFO]           [Section 3.7] Communication protocol schema (PTCF-mapped):
{
  "sender_id": "string \u2014 Agent's unique identifier (Persona)",
  "recipient_id": "string \u2014 Target agent identifier",
  "message_type": "string \u2014 Purpose of message (Task)",
  "timestamp": "ISO 8601 \u2014 When the message was sent (Context)",
  "confidence_score": "float [0-1] \u2014 Agent's confidence in its analysis",
  "data_payload": "object \u2014 Structured content of the message",
  "context_references": "array \u2014 Links to prior messages (Context)",
  "requires_response": "boolean \u2014 Whether a reply is expected",
  "priority_level": "string \u2014 low/medium/high/critical (Context)"
}


[INFO]           [Section 3.7] Simulating: agent_alpha (Credit Risk) → agent_beta (Market Risk)
[SUCCESS]        [Section 3.7] Inter-agent message generated.

--- AGENT-TO-AGENT MESSAGE ---
```json
{
  "sender_id": "agent_alpha",
  "recipient_id": "agent_beta",
  "message_type": "risk_assessment_u

---
## Case Studies: PTCF in Production Contexts

The following three case studies demonstrate how the PTCF framework operates in representative production scenarios. Each uses `@graceful_fallback` and `ColorLogger` for resilient, observable execution.

In [29]:
# ============================================================
# Case Study 1: SaaS Customer Support Triage Agent
# ============================================================

logger_cs1 = ColorLogger(context="Case Study 1")

triage_system = (
    "[PERSONA] You are a Level-2 technical support triage specialist "
    "with authority to assign severity tags and route tickets.\n\n"
    "[TASK] Classify incoming tickets by: (1) category, (2) severity level "
    "(Severity-1: 1hr SLA, Severity-2: 4hr, Severity-3: 24hr), and "
    "(3) route to the appropriate queue.\n\n"
    "[CONTEXT] You serve Fortune 500 clients under strict SLA agreements. "
    "Severity-1 issues must be routed within minutes.\n\n"
    "[FORMAT] Respond with a JSON object: ticket_id, category, severity, "
    "assigned_queue, suggested_action, sla_deadline, reasoning."
)

test_ticket = (
    "Dashboard load times have degraded from 2 seconds to 45 seconds "
    "over the past hour. Multiple teams are affected. Please route "
    "this to the appropriate severity queue."
)

logger_cs1.info(f"Incoming ticket: '{test_ticket[:60]}...'")

triage_prompt = ChatPromptTemplate.from_messages([
    ("system", triage_system),
    ("human", "{ticket}")
])

try:
    triage_chain = triage_prompt | llm
    result = triage_chain.invoke({"ticket": test_ticket})
    logger_cs1.success("Ticket classified and routed.")
    print("\n--- TRIAGE OUTPUT ---")
    print(result.content)
except Exception as e:
    logger_cs1.error(f"Triage failed: {e}")

[INFO]           [Case Study 1] Incoming ticket: 'Dashboard load times have degraded from 2 seconds to 45 seco...'
[SUCCESS]        [Case Study 1] Ticket classified and routed.

--- TRIAGE OUTPUT ---
```json
{
  "ticket_id": "12345",
  "category": "Performance Degradation",
  "severity": "Severity-1",
  "assigned_queue": "Performance Team",
  "suggested_action": "Investigate server load and database performance; check for any recent code deployments.",
  "sla_deadline": "1 hour from ticket creation",
  "reasoning": "The issue affects multiple teams and is a significant degradation in performance, requiring immediate attention to meet SLA commitments."
}
```


In [30]:
# ============================================================
# Case Study 2: Financial Compliance Review Agent
# ============================================================

logger_cs2 = ColorLogger(context="Case Study 2")

compliance_system = (
    "[PERSONA] You are a compliance pre-screening specialist. "
    "You have no authority to provide financial advice — only to flag "
    "potential policy references for human review.\n\n"
    "[TASK] Two-pass process: (1) identify policy-relevant language, "
    "(2) classify risk level (Low, Medium, High).\n\n"
    "[CONTEXT] Applicable regulations: MiFID II, FCA COBS. "
    "Explicit prohibitions: no advice, no predictions. "
    "High-risk classifications must be escalated to a human reviewer.\n\n"
    "[FORMAT] JSON report with: review_id, risk_level, flagged_passages "
    "(array), reasoning, escalation_required."
)

sample_communication = (
    "Dear Client, based on our analysis, this investment is expected to "
    "yield 15% returns over the next fiscal year. We recommend moving "
    "all assets into our premium growth fund for maximum returns."
)

logger_cs2.info("Pre-screening client communication for compliance...")

compliance_prompt = ChatPromptTemplate.from_messages([
    ("system", compliance_system),
    ("human", "Pre-screen the following communication and flag any MiFID or FCA concerns: {text}")
])

try:
    comp_chain = compliance_prompt | llm
    result = comp_chain.invoke({"text": sample_communication})
    logger_cs2.success("Compliance review complete.")
    print("\n--- COMPLIANCE REVIEW OUTPUT ---")
    print(result.content)
except Exception as e:
    logger_cs2.error(f"Compliance review failed: {e}")

[INFO]           [Case Study 2] Pre-screening client communication for compliance...
[SUCCESS]        [Case Study 2] Compliance review complete.

--- COMPLIANCE REVIEW OUTPUT ---
```json
{
  "review_id": "001",
  "risk_level": "High",
  "flagged_passages": [
    "based on our analysis, this investment is expected to yield 15% returns over the next fiscal year",
    "We recommend moving all assets into our premium growth fund for maximum returns"
  ],
  "reasoning": "The communication includes explicit predictions about investment returns, which is a concern under MiFID II and FCA COBS regulations as it may be construed as providing financial advice. Additionally, recommending moving all assets into a specific fund suggests advisory activity.",
  "escalation_required": true
}
```


In [31]:
# ============================================================
# Case Study 3: Automated Code Review Agent
# ============================================================

logger_cs3 = ColorLogger(context="Case Study 3")

codereview_system = (
    "[PERSONA] You are a senior software engineer and code reviewer "
    "with a constructive, non-blocking communication style.\n\n"
    "[TASK] Review code for: (1) style guide compliance, "
    "(2) OWASP Top 10 security patterns, (3) test coverage gaps. "
    "Flag Critical and Major issues only — not cosmetic suggestions.\n\n"
    "[CONTEXT] Language: Python 3.11. Never block a pull request on Minor issues. "
    "Security vulnerabilities take priority.\n\n"
    "[FORMAT] Markdown table with columns: #, Category, Severity, Line Ref, "
    "Recommendation. End with overall_verdict: Approve or Request Changes."
)

sample_code = '''def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = db.execute(query)
    return result

def upload_file(file):
    with open(f"/uploads/{file.name}", "wb") as f:
        f.write(file.read())
    return "uploaded"

def connect_db():
    conn = database.connect(host="prod-db", port=5432)
    return conn'''

logger_cs3.info("Reviewing Python code snippet for security and quality...")

code_prompt = ChatPromptTemplate.from_messages([
    ("system", codereview_system),
    ("human", "Review the following pull request code for security and quality issues:\n\n{code}")
])

try:
    code_chain = code_prompt | llm
    result = code_chain.invoke({"code": sample_code})
    logger_cs3.success("Code review complete.")
    print("\n--- CODE REVIEW OUTPUT ---")
    print(result.content)
except Exception as e:
    logger_cs3.error(f"Code review failed: {e}")

[INFO]           [Case Study 3] Reviewing Python code snippet for security and quality...
[SUCCESS]        [Case Study 3] Code review complete.

--- CODE REVIEW OUTPUT ---
| #  | Category        | Severity | Line Ref | Recommendation                                                                                  |
|----|-----------------|----------|----------|--------------------------------------------------------------------------------------------------|
| 1  | Security        | Critical | 2        | Use parameterized queries to prevent SQL injection vulnerabilities. Consider using prepared statements with placeholders. |
| 2  | Security        | Critical | 7        | The file upload process should validate the file name and sanitize the path to prevent directory traversal attacks. Consider using a secure library or method to handle uploads safely. |
| 3  | Security        | Major    | 12       | Avoid hardcoding sensitive information such as database connection details. Use enviro